In [1]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 requests==2.32.3 beautifulsoup4==4.12.3 lxml==5.3.0 openpyxl==3.1.5 rapidfuzz==3.9.7 tqdm==4.66.5


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time
import unicodedata

from bs4 import BeautifulSoup
import pandas as pd
import requests
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm


C:\Users\Giovanny\anaconda3\envs\fluidgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CWD = Path.cwd().resolve()
BASE_DIR = CWD if CWD.name == "notebooks" else CWD / "notebooks"
OUTPUT_DIR = BASE_DIR / "output"
CACHE_DIR = BASE_DIR / "cache"
RAW_DIR = BASE_DIR / "raw"
REPORTS_DIR = BASE_DIR / "reports"
SAT_RAW_DIR = RAW_DIR / "sat_69b"
SANCIONADOS_RAW_DIR = RAW_DIR / "sancionados"
COMPRANET_RAW_DIR = RAW_DIR / "compranet"
DECLARANET_RAW_DIR = RAW_DIR / "declaranet"

for directory in [BASE_DIR, OUTPUT_DIR, CACHE_DIR, RAW_DIR, REPORTS_DIR, SAT_RAW_DIR, SANCIONADOS_RAW_DIR, COMPRANET_RAW_DIR, DECLARANET_RAW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FORCE_DOWNLOAD = False
ENABLE_LARGE_DOWNLOADS = False
MAX_DOWNLOAD_BYTES = 80 * 1024 * 1024
REQUEST_TIMEOUT = 45
HEADERS = {"User-Agent": "pep-public-intel-notebook/1.0 public-osint-research contact: local-notebook"}
print(f"BASE_DIR: {BASE_DIR}")


BASE_DIR: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks


In [4]:
INPUT_CONTEXT_COLUMNS = ["estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input"]
EVIDENCE_COLUMNS = [
    "seed_id", "nombre_persona_input", "normalized_name_input", "estado_input", "municipio_input",
    "puesto_input", "dependencia_input", "periodo_inicio_input", "periodo_fin_input", "partido_input",
    "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
    "source", "source_type", "source_country", "source_official", "source_category", "source_reliability",
    "matched_record_name", "matched_record_normalized_name", "matched_alias", "matched_entity_type",
    "matched_role", "matched_position", "matched_agency", "matched_state", "matched_municipality",
    "matched_country", "matched_identifier", "matched_company", "matched_rfc", "matched_curp",
    "match_score_pct", "match_strength", "match_method", "matched_fields", "conflicting_fields",
    "requires_review", "identity_confidence_pct", "signal_type", "signal_category", "severity",
    "risk_layer", "risk_level_candidate", "confidence_pct", "evidence_title", "evidence_summary",
    "evidence_snippet", "evidence_keywords", "evidence_date", "evidence_url", "source_url",
    "search_query", "search_engine", "retrieved_at", "raw_file_path", "raw_file_sha256",
    "involvement_summary", "explanation", "limitations", "recommended_action",
]
SUMMARY_COLUMNS = [
    "seed_id", "nombre_persona_input", "sat_69b_candidates", "sancionados_candidates", "compranet_candidates",
    "declaranet_candidates", "official_mexico_signals", "max_match_score_pct", "max_identity_confidence_pct",
    "requires_review", "top_evidence_url", "top_involvement_summary", "recommended_action",
]
PARTICLES = {"de", "del", "la", "las", "los", "el", "y"}


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def remove_accents(value):
    if pd.isna(value):
        return ""
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(ch))


def normalize_text(value):
    text = remove_accents(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_name(value, remove_particles=False):
    tokens = normalize_text(value).split()
    if remove_particles:
        tokens = [token for token in tokens if token not in PARTICLES]
    return " ".join(tokens)


def stable_hash(value):
    return hashlib.sha256(normalize_name(value).encode("utf-8")).hexdigest()[:16]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def score_name_match(left, right):
    left_norm = normalize_name(left)
    right_norm = normalize_name(right)
    if not left_norm or not right_norm:
        return 0.0
    left_tokens = [token for token in left_norm.split() if token not in PARTICLES]
    right_tokens = [token for token in right_norm.split() if token not in PARTICLES]
    if not left_tokens or not right_tokens:
        return 0.0
    left_set = set(left_tokens); right_set = set(right_tokens)
    overlap = len(left_set & right_set)
    if overlap == 0:
        return 0.0
    smaller = min(len(left_set), len(right_set)); larger = max(len(left_set), len(right_set))
    overlap_ratio = overlap / smaller if smaller else 0.0
    scores = [fuzz.WRatio(left_norm, right_norm), fuzz.token_sort_ratio(left_norm, right_norm), fuzz.ratio(left_norm, right_norm)]
    if smaller >= 2 and overlap_ratio >= 0.67:
        scores.append(fuzz.token_set_ratio(left_norm, right_norm))
    raw_score = float(max(scores))
    if smaller == 1 and larger >= 3:
        raw_score = min(raw_score, 59.0)
    elif smaller == 2 and larger >= 4:
        raw_score = min(raw_score, 74.0)
    if overlap_ratio < 0.50:
        raw_score = min(raw_score, 59.0)
    elif overlap_ratio < 0.67:
        raw_score = min(raw_score, 74.0)
    return round(raw_score, 2)


def classify_match(score):
    score = float(score or 0)
    if score >= 95:
        return "exact_or_very_strong_name"
    if score >= 88:
        return "strong_candidate"
    if score >= 75:
        return "medium_candidate"
    if score >= 60:
        return "weak_candidate"
    return "discard"


def empty_evidence_df():
    return pd.DataFrame(columns=EVIDENCE_COLUMNS)


def input_has_only_name(seed):
    return bool(seed.get("has_only_name", False)) or not any(str(seed.get(col, "")).strip() for col in INPUT_CONTEXT_COLUMNS)


def identity_confidence(score, seed, matched_fields, sensitive=False):
    confidence = float(score or 0) * 0.72
    extras = [x.strip() for x in str(matched_fields).split(";") if x.strip() and x.strip() != "nombre"]
    confidence += min(len(extras), 5) * 6
    if sensitive:
        confidence -= 8
    if input_has_only_name(seed):
        confidence = min(confidence, 75)
    if float(score or 0) < 95:
        confidence = min(confidence, 86)
    return round(max(0, min(confidence, 100)), 2)


def method_for(seed_name, candidate_name, manual=True):
    if normalize_name(seed_name) == normalize_name(candidate_name):
        return "exact_normalized_name"
    if manual:
        return "manual_import_match"
    return "token_sort_ratio"


In [5]:
def fallback_seed_dataframe():
    print("ADVERTENCIA: no existe output/00_peps_normalized.csv; se crean datos minimos de prueba para esta corrida.")
    names = ["Claudia Sheinbaum Pardo", "Andres Manuel Lopez Obrador", "Marcelo Ebrard Casaubon", "Adan Augusto Lopez Hernandez", "Ricardo Monreal Avila", "Xochitl Galvez Ruiz", "Samuel Garcia Sepulveda", "Clara Brugada Molina", "Omar Garcia Harfuch", "Luisa Maria Alcalde Lujan"]
    return pd.DataFrame([{"seed_id": stable_hash(name), "nombre_persona_input": name, "normalized_name_input": normalize_name(name), "estado_input": "", "municipio_input": "", "puesto_input": "", "dependencia_input": "", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "has_only_name": True} for name in names])

input_file = OUTPUT_DIR / "00_peps_normalized.csv"
seeds_df = pd.read_csv(input_file, encoding="utf-8-sig").fillna("") if input_file.exists() else fallback_seed_dataframe()
for col in ["seed_id", "nombre_persona_input", "normalized_name_input"] + INPUT_CONTEXT_COLUMNS + ["has_only_name"]:
    if col not in seeds_df.columns:
        seeds_df[col] = ""
print(f"Personas a consultar: {len(seeds_df):,}")
display(seeds_df.head(20))


Personas a consultar: 10


,seed_id,nombre_persona_input,normalized_name_input,token_sort_name,name_tokens,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,rfc_input,curp_input,empresa_relacionada_input,alias_input,has_only_name,input_quality_score,created_at
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,claudia pardo sheinbaum,claudia|sheinbaum|pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024.0,,,,,,,False,0.77,2026-05-07T07:51:29.008342+00:00
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,andres lopez manuel obrador,andres|manuel|lopez|obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018.0,2024.0,,,,,AMLO,False,0.99,2026-05-07T07:51:29.008342+00:00
2,55c8819cc4622591,Marcelo Ebrard Casaubon,marcelo ebrard casaubon,casaubon ebrard marcelo,marcelo|ebrard|casaubon,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,adan augusto hernandez lopez,adan|augusto|lopez|hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00
4,2597a81eb6062feb,Ricardo Monreal Avila,ricardo monreal avila,avila monreal ricardo,ricardo|monreal|avila,Zacatecas,,Legislador / figura publica,Congreso de la Union,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
5,39162acdaa914138,Xochitl Galvez Ruiz,xochitl galvez ruiz,galvez ruiz xochitl,xochitl|galvez|ruiz,Nacional,,Senadora / figura publica,Senado de la Republica,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,samuel garcia sepulveda,garcia samuel sepulveda,samuel|garcia|sepulveda,Nuevo Leon,,Gobernador,Gobierno de Nuevo Leon,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
7,cba9b1a8ead15ac3,Clara Brugada Molina,clara brugada molina,brugada clara molina,clara|brugada|molina,Ciudad de Mexico,,Jefa de Gobierno,Gobierno de la Ciudad de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
8,69573f0325299b8b,Omar Garcia Harfuch,omar garcia harfuch,garcia harfuch omar,omar|garcia|harfuch,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,luisa maria alcalde lujan,alcalde luisa lujan maria,luisa|maria|alcalde|lujan,Nacional,,Figura publica,Gobierno de Mexico,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00


In [6]:
SAT_69B_URLS = []
SANCIONADOS_URLS = []
COMPRANET_URLS = []
DECLARANET_URLS = []

print("Carga manual permitida si no hay URL directa estable o si la descarga falla:")
print(f"- SAT 69-B: coloca CSV/XLSX oficiales en {SAT_RAW_DIR}")
print(f"- Servidores publicos sancionados: coloca CSV/XLSX en {SANCIONADOS_RAW_DIR}")
print(f"- CompraNet/contratos: coloca CSV/XLSX en {COMPRANET_RAW_DIR}; ENABLE_LARGE_DOWNLOADS={ENABLE_LARGE_DOWNLOADS}")
print(f"- DeclaraNet/listados publicos: coloca CSV/XLSX en {DECLARANET_RAW_DIR}")

benchmark_rows = []
for source_name, raw_dir, urls in [
    ("SAT 69-B", SAT_RAW_DIR, SAT_69B_URLS),
    ("Servidores publicos sancionados", SANCIONADOS_RAW_DIR, SANCIONADOS_URLS),
    ("CompraNet", COMPRANET_RAW_DIR, COMPRANET_URLS if ENABLE_LARGE_DOWNLOADS else []),
    ("DeclaraNet", DECLARANET_RAW_DIR, DECLARANET_URLS),
]:
    status = "manual_only"
    note = f"Sin URL directa estable configurada; usar {raw_dir}"
    if source_name == "CompraNet" and not ENABLE_LARGE_DOWNLOADS:
        status = "skipped_by_default"; note = "ENABLE_LARGE_DOWNLOADS=False; usar carga manual en raw/compranet"
    benchmark_rows.append({"notebook": "02_fuentes_mexico", "source": source_name, "step": "download", "duration_seconds": 0.0, "records_processed": 0, "records_per_second": 0, "status": status, "error_message": "", "notes": note})


Carga manual permitida si no hay URL directa estable o si la descarga falla:
- SAT 69-B: coloca CSV/XLSX oficiales en C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\raw\sat_69b
- Servidores publicos sancionados: coloca CSV/XLSX en C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\raw\sancionados
- CompraNet/contratos: coloca CSV/XLSX en C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\raw\compranet; ENABLE_LARGE_DOWNLOADS=False
- DeclaraNet/listados publicos: coloca CSV/XLSX en C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\raw\declaranet


In [7]:
def list_tabular_files(raw_dir):
    files = []
    for pattern in ["*.csv", "*.CSV", "*.xlsx", "*.XLSX", "*.xls", "*.XLS"]:
        files.extend(raw_dir.glob(pattern))
    return sorted(set(files))


def read_csv_safely(path):
    last_error = None
    for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=encoding, sep=None, engine="python", dtype=str)
        except Exception as exc:
            last_error = exc
    raise last_error


def read_tabular_files(raw_dir, source_name):
    start = time.perf_counter()
    frames = []
    errors = []
    files = list_tabular_files(raw_dir)
    for path in files:
        try:
            if path.suffix.lower() == ".csv":
                df = read_csv_safely(path)
                df["_source_file"] = path.name
                df["_source_sheet"] = ""
                df["_raw_file_path"] = str(path)
                df["_raw_file_sha256"] = sha256_file(path)
                frames.append(df.fillna(""))
            else:
                sheets = pd.read_excel(path, dtype=str, sheet_name=None)
                for sheet_name, df in sheets.items():
                    df["_source_file"] = path.name
                    df["_source_sheet"] = sheet_name
                    df["_raw_file_path"] = str(path)
                    df["_raw_file_sha256"] = sha256_file(path)
                    frames.append(df.fillna(""))
        except Exception as exc:
            errors.append(f"{path.name}: {str(exc)[:160]}")
    elapsed = time.perf_counter() - start
    rows = sum(len(frame) for frame in frames)
    status = "ok" if frames else "no_files" if not files else "no_readable_files"
    benchmark_rows.append({"notebook": "02_fuentes_mexico", "source": source_name, "step": "parsing", "duration_seconds": round(elapsed, 6), "records_processed": rows, "records_per_second": round(rows / elapsed, 2) if elapsed else 0, "status": status, "error_message": "; ".join(errors)[:500], "notes": f"files={len(files)}"})
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def find_column(columns, patterns):
    normalized = {column: normalize_text(column) for column in columns}
    for pattern in patterns:
        p = normalize_text(pattern)
        tokens = p.split()
        for column, norm in normalized.items():
            if p and p in norm:
                return column
            if tokens and all(token in norm for token in tokens):
                return column
    return None


def compact_row_text(row, max_chars=900):
    parts = []
    for key, value in row.items():
        value = str(value).strip()
        if value and not str(key).startswith("_"):
            parts.append(f"{key}={value}")
    return " | ".join(parts)[:max_chars]


In [8]:
SOURCE_CONFIGS = {
    "sat_69b": {"label": "SAT 69-B", "raw_dir": SAT_RAW_DIR, "source_type": "official_mexico_fiscal_list", "source_category": "mexico_public_fiscal", "signal_type": "sat_69b_candidate", "risk_layer": "tax_public_record", "name_patterns": ["razon social", "nombre", "contribuyente", "denominacion"], "id_patterns": ["rfc"], "state_patterns": ["estado", "entidad"], "agency_patterns": ["autoridad", "dependencia"], "output": OUTPUT_DIR / "02_sat_69b_evidence.csv"},
    "sancionados": {"label": "Servidores publicos sancionados", "raw_dir": SANCIONADOS_RAW_DIR, "source_type": "official_mexico_sanctions_registry", "source_category": "mexico_public_sanctions", "signal_type": "servidor_publico_sancionado_candidate", "risk_layer": "administrative_sanctions", "name_patterns": ["nombre", "servidor publico", "persona sancionada", "funcionario"], "id_patterns": ["rfc", "curp", "expediente"], "state_patterns": ["estado", "entidad"], "agency_patterns": ["dependencia", "institucion", "autoridad"], "output": OUTPUT_DIR / "02_sancionados_evidence.csv"},
    "compranet": {"label": "CompraNet contratos publicos", "raw_dir": COMPRANET_RAW_DIR, "source_type": "official_mexico_contracts", "source_category": "mexico_public_contracts", "signal_type": "proveedor_nombre_pep_candidate", "risk_layer": "public_contracts", "name_patterns": ["proveedor", "contratista", "razon social", "nombre", "representante"], "id_patterns": ["rfc", "contrato", "codigo expediente"], "state_patterns": ["estado", "entidad"], "agency_patterns": ["dependencia", "unidad compradora", "institucion"], "output": OUTPUT_DIR / "02_compranet_evidence.csv"},
    "declaranet": {"label": "DeclaraNet", "raw_dir": DECLARANET_RAW_DIR, "source_type": "official_mexico_public_disclosure", "source_category": "mexico_public_disclosure", "signal_type": "declaranet_record_candidate", "risk_layer": "asset_disclosure", "name_patterns": ["nombre", "servidor publico", "declarante", "funcionario"], "id_patterns": ["rfc", "curp"], "state_patterns": ["estado", "entidad"], "agency_patterns": ["dependencia", "institucion", "cargo", "puesto"], "output": OUTPUT_DIR / "02_declaranet_evidence.csv"},
}

source_data = {}
for key, cfg in SOURCE_CONFIGS.items():
    source_data[key] = read_tabular_files(cfg["raw_dir"], cfg["label"])
    print(f"{cfg['label']}: filas leidas={len(source_data[key]):,}")


SAT 69-B: filas leidas=0
Servidores publicos sancionados: filas leidas=0
CompraNet contratos publicos: filas leidas=0
DeclaraNet: filas leidas=0


In [9]:
def candidate_rows(df, cfg):
    if df.empty:
        return pd.DataFrame()
    name_col = find_column(df.columns, cfg["name_patterns"])
    if not name_col:
        return pd.DataFrame()
    id_col = find_column(df.columns, cfg.get("id_patterns", []))
    state_col = find_column(df.columns, cfg.get("state_patterns", []))
    agency_col = find_column(df.columns, cfg.get("agency_patterns", []))
    rows = []
    for idx, row in df.fillna("").iterrows():
        name = str(row.get(name_col, "")).strip()
        if not name:
            continue
        rows.append({
            "row_index": idx,
            "candidate_name": name,
            "normalized_candidate_name": normalize_name(name),
            "matched_identifier": str(row.get(id_col, "")) if id_col else "",
            "matched_state": str(row.get(state_col, "")) if state_col else "",
            "matched_agency": str(row.get(agency_col, "")) if agency_col else "",
            "row_text": compact_row_text(row),
            "raw_file_path": row.get("_raw_file_path", ""),
            "raw_file_sha256": row.get("_raw_file_sha256", ""),
            "source_file": row.get("_source_file", ""),
        })
    return pd.DataFrame(rows)


def compare_context(seed, cand):
    matched = ["nombre"]
    conflicts = []
    for input_col, cand_col, label in [("estado_input", "matched_state", "estado"), ("dependencia_input", "matched_agency", "dependencia")]:
        i = normalize_text(seed.get(input_col, "")); c = normalize_text(cand.get(cand_col, ""))
        if i and c and (i in c or c in i):
            matched.append(label)
        elif i and c and label == "estado":
            conflicts.append(f"{input_col}={seed.get(input_col, '')} pero {cand_col}={cand.get(cand_col, '')}")
    if seed.get("rfc_input") and cand.get("matched_identifier") and normalize_text(seed.get("rfc_input")) in normalize_text(cand.get("matched_identifier")):
        matched.append("rfc")
    return "; ".join(matched), "; ".join(conflicts)


def evidence_row(seed, cand, cfg, score, source_key):
    matched_fields, conflicts = compare_context(seed, cand)
    sensitive = source_key in {"sat_69b", "sancionados"}
    id_conf = identity_confidence(score, seed, matched_fields, sensitive=sensitive)
    signal_type = cfg["signal_type"]
    row_text_norm = normalize_text(cand.get("row_text", ""))
    if source_key == "compranet" and "adjudicacion" in row_text_norm and "directa" in row_text_norm:
        signal_type = "adjudicacion_directa_candidate"
    elif source_key == "compranet" and seed.get("empresa_relacionada_input"):
        signal_type = "empresa_relacionada_contrato_publico_candidate"
    risk = "medium_review" if score >= 88 else "low_signal"
    action = "verify_identity" if score >= 88 else "manual_review" if score >= 75 else "discard_likely_false_positive"
    involvement = {
        "sat_69b": "Coincidencia nominal en senal fiscal publica; no implica delito y requiere revision contextual.",
        "sancionados": "Coincidencia nominal en registro publico de sanciones administrativas; requiere verificacion de identidad.",
        "compranet": "Proveedor con nombre similar aparece en contratos publicos; no se confirma identidad personal.",
        "declaranet": "Registro publico de declaracion patrimonial asociado a nombre y/o dependencia similar.",
    }.get(source_key, "Coincidencia nominal en fuente publica mexicana; requiere revision.")
    return {
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""), "normalized_name_input": seed.get("normalized_name_input", ""),
        "estado_input": seed.get("estado_input", ""), "municipio_input": seed.get("municipio_input", ""), "puesto_input": seed.get("puesto_input", ""),
        "dependencia_input": seed.get("dependencia_input", ""), "periodo_inicio_input": seed.get("periodo_inicio_input", ""), "periodo_fin_input": seed.get("periodo_fin_input", ""),
        "partido_input": seed.get("partido_input", ""), "rfc_input": seed.get("rfc_input", ""), "curp_input": seed.get("curp_input", ""),
        "empresa_relacionada_input": seed.get("empresa_relacionada_input", ""), "alias_input": seed.get("alias_input", ""),
        "source": cfg["label"], "source_type": cfg["source_type"], "source_country": "Mexico", "source_official": True,
        "source_category": cfg["source_category"], "source_reliability": "official_or_manual_bulk_file",
        "matched_record_name": cand.get("candidate_name", ""), "matched_record_normalized_name": cand.get("normalized_candidate_name", ""), "matched_alias": "",
        "matched_entity_type": "person_or_company", "matched_role": "", "matched_position": "", "matched_agency": cand.get("matched_agency", ""),
        "matched_state": cand.get("matched_state", ""), "matched_municipality": "", "matched_country": "Mexico", "matched_identifier": cand.get("matched_identifier", ""),
        "matched_company": cand.get("candidate_name", "") if source_key in {"sat_69b", "compranet"} else "", "matched_rfc": cand.get("matched_identifier", "") if "rfc" in normalize_text(cand.get("matched_identifier", "")) else "", "matched_curp": "",
        "match_score_pct": score, "match_strength": classify_match(score), "match_method": method_for(seed.get("nombre_persona_input", ""), cand.get("candidate_name", ""), manual=True),
        "matched_fields": matched_fields, "conflicting_fields": conflicts, "requires_review": True, "identity_confidence_pct": id_conf,
        "signal_type": signal_type, "signal_category": cfg["source_category"], "severity": "official_mexico_public_record_candidate",
        "risk_layer": cfg["risk_layer"], "risk_level_candidate": risk, "confidence_pct": id_conf,
        "evidence_title": f"{cfg['label']} candidate: {cand.get('candidate_name', '')}", "evidence_summary": cand.get("row_text", ""),
        "evidence_snippet": cand.get("row_text", ""), "evidence_keywords": cfg["source_category"], "evidence_date": "",
        "evidence_url": "manual_or_bulk_public_file", "source_url": "manual_or_bulk_public_file", "search_query": seed.get("nombre_persona_input", ""), "search_engine": "",
        "retrieved_at": now_utc(), "raw_file_path": cand.get("raw_file_path", ""), "raw_file_sha256": cand.get("raw_file_sha256", ""),
        "involvement_summary": involvement, "explanation": "Candidato de coincidencia por similitud nominal y campos disponibles. No constituye confirmacion de identidad ni conclusion legal.",
        "limitations": "La calidad depende de columnas presentes en el archivo cargado. Homonimos y registros empresariales requieren verificacion manual.", "recommended_action": action,
    }


def match_source(source_key, df, cfg):
    start = time.perf_counter()
    candidates = candidate_rows(df, cfg)
    if candidates.empty:
        elapsed = time.perf_counter() - start
        benchmark_rows.append({"notebook": "02_fuentes_mexico", "source": cfg["label"], "step": "matching", "duration_seconds": round(elapsed, 6), "records_processed": 0, "records_per_second": 0, "status": "no_candidates", "error_message": "", "notes": f"Coloca CSV/XLSX con columna de nombre en {cfg['raw_dir']}"})
        return empty_evidence_df()
    choices = candidates["normalized_candidate_name"].tolist()
    rows = []
    comparisons = len(seeds_df) * len(candidates)
    for _, seed in tqdm(seeds_df.iterrows(), total=len(seeds_df), desc=f"Matching {cfg['label']}"):
        query_terms = [(seed.get("nombre_persona_input", ""), "nombre")]
        if seed.get("empresa_relacionada_input"):
            query_terms.append((seed.get("empresa_relacionada_input", ""), "empresa_relacionada"))
        for query, _kind in query_terms:
            extracted = process.extract(normalize_name(query), choices, scorer=fuzz.WRatio, score_cutoff=60, limit=25)
            for _, _rough, idx in extracted:
                cand = candidates.iloc[idx]
                score = score_name_match(query, cand.get("candidate_name", ""))
                if classify_match(score) == "discard":
                    continue
                rows.append(evidence_row(seed, cand, cfg, score, source_key))
    elapsed = time.perf_counter() - start
    benchmark_rows.append({"notebook": "02_fuentes_mexico", "source": cfg["label"], "step": "matching", "duration_seconds": round(elapsed, 6), "records_processed": int(comparisons), "records_per_second": round(comparisons / elapsed, 2) if elapsed else 0, "status": "ok", "error_message": "", "notes": f"matches={len(rows)}; candidates={len(candidates)}"})
    evidence = pd.DataFrame(rows, columns=EVIDENCE_COLUMNS) if rows else empty_evidence_df()
    return evidence.drop_duplicates() if not evidence.empty else evidence


In [10]:
source_evidence = {}
for key, cfg in SOURCE_CONFIGS.items():
    evidence = match_source(key, source_data[key], cfg)
    source_evidence[key] = evidence
    evidence.to_csv(cfg["output"], index=False, encoding="utf-8-sig")
    print(f"{cfg['label']}: evidence_rows={len(evidence):,}; archivo={cfg['output']}")

consolidated_df = pd.concat(list(source_evidence.values()), ignore_index=True) if source_evidence else empty_evidence_df()
if consolidated_df.empty:
    consolidated_df = empty_evidence_df()
consolidated_path = OUTPUT_DIR / "02_fuentes_mexico_evidence.csv"
consolidated_df.to_csv(consolidated_path, index=False, encoding="utf-8-sig")
print(f"Consolidado fuentes Mexico: {consolidated_path}")
display(consolidated_df.head(20))


SAT 69-B: evidence_rows=0; archivo=C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_sat_69b_evidence.csv
Servidores publicos sancionados: evidence_rows=0; archivo=C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_sancionados_evidence.csv
CompraNet contratos publicos: evidence_rows=0; archivo=C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_compranet_evidence.csv
DeclaraNet: evidence_rows=0; archivo=C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_declaranet_evidence.csv
Consolidado fuentes Mexico: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_fuentes_mexico_evidence.csv


,seed_id,nombre_persona_input,normalized_name_input,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,...,source_url,search_query,search_engine,retrieved_at,raw_file_path,raw_file_sha256,involvement_summary,explanation,limitations,recommended_action


In [11]:
summary_rows = []
for _, seed in seeds_df.iterrows():
    subset = consolidated_df[consolidated_df["seed_id"].astype(str) == str(seed.get("seed_id", ""))] if not consolidated_df.empty else empty_evidence_df()
    counts = {}
    for key, cfg in SOURCE_CONFIGS.items():
        counts[key] = int(subset["source"].astype(str).eq(cfg["label"]).sum()) if not subset.empty else 0
    max_match = pd.to_numeric(subset["match_score_pct"], errors="coerce").fillna(0).max() if len(subset) else 0
    max_conf = pd.to_numeric(subset["identity_confidence_pct"], errors="coerce").fillna(0).max() if len(subset) else 0
    top = None
    if len(subset):
        temp = subset.copy(); temp["_score"] = pd.to_numeric(temp["match_score_pct"], errors="coerce").fillna(0)
        top = temp.sort_values("_score", ascending=False).iloc[0]
    if len(subset) == 0:
        action = "no_action"
    elif max_match >= 88:
        action = "verify_identity"
    else:
        action = "manual_review"
    summary_rows.append({
        "seed_id": seed.get("seed_id", stable_hash(seed.get("nombre_persona_input", ""))),
        "nombre_persona_input": seed.get("nombre_persona_input", ""),
        "sat_69b_candidates": counts.get("sat_69b", 0),
        "sancionados_candidates": counts.get("sancionados", 0),
        "compranet_candidates": counts.get("compranet", 0),
        "declaranet_candidates": counts.get("declaranet", 0),
        "official_mexico_signals": int(len(subset)),
        "max_match_score_pct": round(float(max_match), 2),
        "max_identity_confidence_pct": round(float(max_conf), 2),
        "requires_review": bool(len(subset) > 0 or input_has_only_name(seed)),
        "top_evidence_url": top.get("evidence_url", "") if top is not None else "",
        "top_involvement_summary": top.get("involvement_summary", "") if top is not None else "",
        "recommended_action": action,
    })
summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
summary_path = OUTPUT_DIR / "02_fuentes_mexico_person_summary.csv"
benchmark_path = OUTPUT_DIR / "02_benchmark_fuentes_mexico.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
pd.DataFrame(benchmark_rows).to_csv(benchmark_path, index=False, encoding="utf-8-sig")
print(f"Summary guardado: {summary_path}")
print(f"Benchmark guardado: {benchmark_path}")
display(summary_df)
display(pd.DataFrame(benchmark_rows))


Summary guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_fuentes_mexico_person_summary.csv
Benchmark guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_benchmark_fuentes_mexico.csv


,seed_id,nombre_persona_input,sat_69b_candidates,sancionados_candidates,compranet_candidates,declaranet_candidates,official_mexico_signals,max_match_score_pct,max_identity_confidence_pct,requires_review,top_evidence_url,top_involvement_summary,recommended_action
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,0,0,0,0,0,0.0,0.0,False,,,no_action
1,910a01d167e16711,Andres Manuel Lopez Obrador,0,0,0,0,0,0.0,0.0,False,,,no_action
2,55c8819cc4622591,Marcelo Ebrard Casaubon,0,0,0,0,0,0.0,0.0,False,,,no_action
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,0,0,0,0,0,0.0,0.0,False,,,no_action
4,2597a81eb6062feb,Ricardo Monreal Avila,0,0,0,0,0,0.0,0.0,False,,,no_action
5,39162acdaa914138,Xochitl Galvez Ruiz,0,0,0,0,0,0.0,0.0,False,,,no_action
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,0,0,0,0,0,0.0,0.0,False,,,no_action
7,cba9b1a8ead15ac3,Clara Brugada Molina,0,0,0,0,0,0.0,0.0,False,,,no_action
8,69573f0325299b8b,Omar Garcia Harfuch,0,0,0,0,0,0.0,0.0,False,,,no_action
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,0,0,0,0,0,0.0,0.0,False,,,no_action


,notebook,source,step,duration_seconds,records_processed,records_per_second,status,error_message,notes
0,02_fuentes_mexico,SAT 69-B,download,0.000000,0,0.0,manual_only,,Sin URL directa estable configurada; usar C:\U...
1,02_fuentes_mexico,Servidores publicos sancionados,download,0.000000,0,0.0,manual_only,,Sin URL directa estable configurada; usar C:\U...
2,02_fuentes_mexico,CompraNet,download,0.000000,0,0.0,skipped_by_default,,ENABLE_LARGE_DOWNLOADS=False; usar carga manua...
3,02_fuentes_mexico,DeclaraNet,download,0.000000,0,0.0,manual_only,,Sin URL directa estable configurada; usar C:\U...
4,02_fuentes_mexico,SAT 69-B,parsing,0.001246,0,0.0,no_files,,files=0
5,02_fuentes_mexico,Servidores publicos sancionados,parsing,0.000781,0,0.0,no_files,,files=0
6,02_fuentes_mexico,CompraNet contratos publicos,parsing,0.000682,0,0.0,no_files,,files=0
7,02_fuentes_mexico,DeclaraNet,parsing,0.000699,0,0.0,no_files,,files=0
8,02_fuentes_mexico,SAT 69-B,matching,0.000324,0,0.0,no_candidates,,Coloca CSV/XLSX con columna de nombre en C:\Us...
9,02_fuentes_mexico,Servidores publicos sancionados,matching,0.000180,0,0.0,no_candidates,,Coloca CSV/XLSX con columna de nombre en C:\Us...


In [12]:
print("Resumen notebook 02")
print(f"1. Personas procesadas: {len(seeds_df):,}")
print(f"2. Filas de evidencia generadas: {len(consolidated_df):,}")
print(f"3. Personas con hits: {consolidated_df['seed_id'].nunique() if not consolidated_df.empty else 0:,}")
print("4. Top 10 filas tabulares:")
display(consolidated_df.head(10))
print("5. CSV generados:")
for path in [SOURCE_CONFIGS['sat_69b']['output'], SOURCE_CONFIGS['sancionados']['output'], SOURCE_CONFIGS['compranet']['output'], SOURCE_CONFIGS['declaranet']['output'], consolidated_path, summary_path, benchmark_path]:
    print(f"- {path}")


Resumen notebook 02
1. Personas procesadas: 10
2. Filas de evidencia generadas: 0
3. Personas con hits: 0
4. Top 10 filas tabulares:


,seed_id,nombre_persona_input,normalized_name_input,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,...,source_url,search_query,search_engine,retrieved_at,raw_file_path,raw_file_sha256,involvement_summary,explanation,limitations,recommended_action


5. CSV generados:
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_sat_69b_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_sancionados_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_compranet_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_declaranet_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_fuentes_mexico_evidence.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_fuentes_mexico_person_summary.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\02_benchmark_fuentes_mexico.csv
